In [1]:
import functools
import jax
import os

from datetime import datetime
from jax import numpy as jp
import matplotlib.pyplot as plt

from IPython.display import HTML, clear_output

import brax
import flax
from brax import envs
from brax.io import model
from brax.io import json
from brax.io import html
# from brax.training.agents.ppo import train as ppo
# from brax.training.agents.sac import train as sac

import functools
import time
from typing import Any, Callable, Mapping, Optional, Tuple, Union

from absl import logging
from brax import base
from brax import envs
from brax.training import acting
from brax.training import gradients
from brax.training import pmap
from brax.training import types
from brax.training.acme import running_statistics
from brax.training.acme import specs
from brax.training.agents.ppo import losses as ppo_losses
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.types import Params
from brax.training.types import PRNGKey
from brax.v1 import envs as envs_v1
from etils import epath
import flax
import jax
import jax.numpy as jnp
import numpy as np
import optax
from orbax import checkpoint as ocp

from brax.envs.base import PipelineEnv, State
from brax.io import mjcf
from etils import epath

import wandb
import xmltodict
from time import strftime, gmtime

from ppo import ppo_train
from env import HalfcheetahWithObstacles, HalfcheetahMorphTasks, HalfcheetahMML

import cv2

os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

In [2]:
"""
Option: init wandb separately
"""
wandb.init(
    project='test',
    group='vu',
    name=f'zuxinrui_task',
    mode="online",
)

wandb: Currently logged in as: zuxinrui to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
"""
Option: init wandb separately
"""
wandb.init(
    project='test',
    group='vu',
    name=f'zuxinrui_task',
    mode="online",
)

wandb: Currently logged in as: zuxinrui to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [21]:
obstacle_height = np.random.uniform(0.2, 0.5)
obstacle_width = np.random.uniform(0.1, 0.5)
obstacle_spacing = np.random.uniform(0.5, 2.0)
bth_r, bsh_r, bfo_r, fth_r, fsh_r, ffo_r = np.random.uniform(low=0.5, high=2.0, size=6)

def _torso_pitch(ps: base.State):
    """Compute torso pitch (y-axis)."""
    qw, qx, qy, qz = ps.x.rot[0]
    t2 = 2 * (qw*qy - qz*qx)
    t2 = jp.clip(t2, -1.0, 1.0)
    return jp.arcsin(t2)

def _torso_pitch_unwrapped(ps: base.State):
    # quaternion (w, x, y, z)
    qw, qx, qy, qz = ps.x.rot[0]
    # pitch around the y-axis: atan2 formula gives range −π … +π
    pitch = jp.arctan2(2 * (qw*qy - qx*qz),
                       1 - 2 * (qy*qy + qx*qx))
    return pitch

def _cumulative_pitch(ps, state):
    pitch_now  = _torso_pitch_unwrapped(ps)
    pitch_prev = _torso_pitch_unwrapped(state.pipeline_state)
    # unwrap step-by-step
    delta = pitch_now - pitch_prev
    delta = jp.where(delta >  jp.pi, delta - 2*jp.pi, delta)
    delta = jp.where(delta < -jp.pi, delta + 2*jp.pi, delta)
    return state.metrics['cum_pitch'] + delta, delta      # see metrics update

def reward_function(ps, state, dt):
    # backflip reward
    cum_pitch, delta = _cumulative_pitch(ps, state)
    pitch_dot = (delta / dt)
    # pitch     = _torso_pitch(ps)
    # pitch_dot = (pitch - _torso_pitch(ps0)) / dt
    task_rew  = -0.5 * pitch_dot  #  - 0.1 * cum_pitch
        # flipped   = 15.0 * (jp.abs(cum_pitch) // (2 * jp.pi))  # 2 * pi
    task_rew += 100.0 * (jp.abs(cum_pitch) // (2 * jp.pi))
    task_rew += 20.0 * (jp.abs(cum_pitch) // (jp.pi))
    task_done = False  # flipped
    return task_rew, task_done, cum_pitch


# 1.  unwrapped pitch (−π … +π) for a single step -------------------
def _pitch_step(ps: base.State):
    # (w, x, y, z)
    qw, qx, qy, qz = ps.x.rot[0]
    # y-axis rotation
    return jp.arctan2(2 * (qw*qy - qx*qz),
                      1 - 2 * (qy*qy + qx*qx))

# 2.  cumulative pitch across steps (kept in `metrics`) --------------
def _cum_pitch(ps, state):
    p_now  = _pitch_step(ps)
    p_prev = _pitch_step(state.pipeline_state)

    # unwrap tiny neighbourhood around ±π
    d = p_now - p_prev
    d = jp.where(d >  jp.pi, d - 2*jp.pi, d)
    d = jp.where(d < -jp.pi, d + 2*jp.pi, d)

    return state.metrics['cum_pitch'] + d, d      # cum, delta

# 3.  which of the root geoms touch the ground -----------------------
# def _root_on_floor(ps, thresh=0.02):
#     # contact.pos[..., 2] is penetration depth (negative == penetration)
#     ground_hit = jp.logical_and(ps.contact.pos[..., 2] < thresh,
#                                 ps.contact.geom1 < 6)   # torso geoms < 6
#     return jp.any(ground_hit)

def _root_on_floor(ps, thresh=0.15):          # 15 cm
    return ps.x.pos[0, 2] < thresh

# ------------------------------------------------------------
#  reward function  (use this as your `task_id` callable)
# ------------------------------------------------------------
def backflip_reward(ps: base.State,
                    state,          # previous State
                    dt: float):

    # --- kinematics --------------------------------------------------
    cum_pitch, delta_pitch = _cum_pitch(ps, state)
    pitch_rate  = delta_pitch / dt              # rad / s
    root_h      = ps.x.pos[0, 2]                # torso height (m)
    airborne    = root_h > 0.25                 # 25 cm above ground
    # on_floor    = _root_on_floor(ps)

    # --- shaping terms ----------------------------------------------
    jump_rew    = 2.0 * jp.clip(root_h - 0.25, 0.0, 0.35)   # 0 … 0.7
    # spin_rew    = 0.1 * jp.abs(pitch_rate) * airborne       # only in air
    # spin_rew    = -1.0 * pitch_rate * airborne
    spin_rew    = -1.0 * pitch_rate
    # belly_pen   = jp.where(on_floor, -1.0, 0.0)             # every step

    # --- success bonus + termination -------------------------------
    flipped     = jp.logical_and(airborne,
                                 cum_pitch > 2*jp.pi - 0.3)     # 360°
    n_flip = jp.floor(cum_pitch / (2 * jp.pi))
    bonus = 100 * n_flip
    # upright     = jp.abs(_pitch_step(ps)) < 0.4                # landed
    # success     = jp.logical_and(flipped, upright)
    #
    # bonus       = jp.where(success, 25.0, 0.0)             # sparse

    # --- assemble reward --------------------------------------------
    task_rew    = spin_rew + bonus  #  + belly_pen
    task_done   = False

    # --- deliver extra info back to env.step ------------------------
    #     (env.step will stash cum_pitch & flat_counter in metrics)
    return task_rew, task_done, cum_pitch

# high jump:
def high_jump(ps: base.State,
                    state,          # previous State
                    dt: float):
    root_h   = ps.x.pos[0, 2]
    root_h0  = state.pipeline_state.x.pos[0, 2]

    # in‐air flag (above a small threshold)
    air_now  = root_h  > 0.1
    air_prev = root_h0 > 0.1

    # landed = was in air last step, now on ground
    landed   = jp.logical_and(~air_now, air_prev)

    # dense reward: current height
    dense_rew = root_h

    # sparse bonus on landing for having jumped
    bonus     = jp.where(landed, 5.0, 0.0)

    task_rew  = dense_rew + bonus
    task_done = False
    return task_rew, task_done, 0.0

# high jump:
def high_jump_velocity(ps: base.State,
                    state,          # previous State
                    dt: float):
    root_h   = ps.x.pos[0, 2]
    root_h0  = state.pipeline_state.x.pos[0, 2]

    v = root_h - root_h0 / dt
    task_rew = 0.1 * jp.abs(v)
    task_done = False
    return task_rew, task_done, 0.0

def _is_airborne(ps: base.State, thresh=0.05):
    """Both feet higher than thresh."""
    z_back  = ps.x.pos[2, 2]   # geom 2
    z_front = ps.x.pos[5, 2]   # geom 5
    return jp.logical_and(z_back > thresh, z_front > thresh)

def long_jump(ps: base.State,
                    state,          # previous State
                    dt: float):
    # 1) read root x‐ and z‐positions now & one step ago
    x,  z  = ps.x.pos[0, 0], ps.x.pos[0, 2]
    x0, z0 = state.pipeline_state.x.pos[0, 0], state.pipeline_state.x.pos[0, 2]
    weight_air_dx = 2.0
    weight_air = 0.0

    # 2) decide if we are airborne (above 10 cm)
    # air_now  = z  > 0.10
    # air_prev = z0 > 0.10
    air_now  = _is_airborne(ps)
    air_prev = _is_airborne(state.pipeline_state)

    # 3) horizontal increment
    dx = x - x0

    # 4) only count that increment while in the air
    air_dx = jp.where(air_now, dx, 0.0)

    # 5) small per-step hang-time bonus
    air_bonus = jp.where(air_now, 1.0, 0.0)

    # 6) landing bonus: you get +5 the step you go from air→ground
    landed = jp.logical_and(~air_now, air_prev)
    land_bonus = jp.where(landed, -100.0, 0.0)

    # assemble
    task_rew  = weight_air_dx * air_dx + weight_air * air_bonus
    task_done = landed

    return task_rew, task_done, 0.0

env = HalfcheetahMML(
    forward_reward_weight=0.0,
    obstacle_height=0.01,  # (0.2 - 0.5) obstacle_height
    obstacle_width=obstacle_width,  # (0.1 - 0.5)
    obstacle_spacing=obstacle_spacing,  # (0.5 - 2.0)
    n_obstacles=0,  # 10
    design=(bth_r, bsh_r, bfo_r, fth_r, fsh_r, ffo_r),
    backend='spring',
    task_id=high_jump_velocity,  # 'backflip'
)

state = jax.jit(env.reset)(rng=jax.random.PRNGKey(seed=0))

url = html.render(env.sys.tree_replace({"opt.timestep": env.dt}), [state.pipeline_state], height=1024)
# with open(os.path.join(exp_dir, f"{exp_name}_{num_steps}.html"), "w") as file:
#     file.write(url)
wandb.log({"env render": wandb.Html(url)})
    # n minibatch really doesn't matter too much!

Changing morphology 0.5159883256649049 0.7673906300598851 0.5258731860322508 0.5628558020478791 0.8147063362295274 0.7088109222827816


In [22]:
episode_length = 150

train_fn = functools.partial(
    ppo_train,
    num_timesteps=30_000_000,
    num_evals=0,
    reward_scaling=1,
    episode_length=episode_length,
    normalize_observations=True,
    action_repeat=1,
    unroll_length=20,
    num_minibatches=32,
    num_updates_per_batch=8,
    discounting=0.95,
    learning_rate=3e-4,
    entropy_cost=0.001,
    num_envs=4096,  # 2048 on 4070 ti s is the fastest  p.s.: num_envs must be divisible by n_batch * batch_size (1+ times per env simulation in the batch)
    batch_size=128,
    seed=3,
)

xdata, ydata = [], []
times = [datetime.now()]

def progress(num_steps, metrics, params, make_policy):
    render(make_policy, params, env, './logs/htmls/', 'halfcheetah', num_steps, metrics)
    times.append(datetime.now())

def render(make_policy, params, env, exp_dir, exp_name, num_steps, metrics=None):
    policy = make_policy(params)
    jit_env_reset = jax.jit(env.reset)
    jit_env_step = jax.jit(env.step)
    jit_policy = jax.jit(policy)

    rollout = []
    key = jax.random.PRNGKey(seed=1)
    key, subkey = jax.random.split(key)
    state = jit_env_reset(rng=subkey)
    for i in range(episode_length):  # 1000 = 50s
        rollout.append(state.pipeline_state)
        key, subkey = jax.random.split(key)
        action, _ = jit_policy(state.obs, subkey)  # Policy requires batched dimension
        # action = action[0]  # Remove batch dimension
        state = jit_env_step(state, action)
        # if i % 1000 == 0:
        #     key, subkey = jax.random.split(key)
        #     state = jit_env_reset(rng=subkey)

    url = html.render(env.sys.tree_replace({"opt.timestep": env.dt}), rollout, height=1024)
    with open(os.path.join(exp_dir, f"{exp_name}_{num_steps}.html"), "w") as file:
        file.write(url)
    wandb.log({
        "video": wandb.Html(url),
        'training/reward': metrics['eval/episode_reward'],
    })

    frames = env.render(rollout, camera='track')
    fps = 30  # Set the desired frames per second of the video
    video_writer = cv2.VideoWriter(
        f'./logs/halfcheetah_{obstacle_height}_{obstacle_width}_{obstacle_spacing}_{env.bth_r}_{env.bsh_r}_{env.bfo_r}_{env.fth_r}_{env.fsh_r}_{env.ffo_r}.mp4', cv2.VideoWriter_fourcc(*"mp4v"), fps, (320, 240)  # remember to switch the x/y axis in cv2
    )
    for i in range(len(frames)):
        video_writer.write(cv2.cvtColor(frames[i], cv2.COLOR_RGB2BGR))
    video_writer.release()

make_inference_fn, params, _ = train_fn(environment=env, progress_fn=progress)

print(f'time overall: {times[-1] - times[0]}')
print(f'height: {obstacle_height}, width: {obstacle_width}, spacing: {obstacle_spacing}')
print(f'design: {env.bth_r, env.bsh_r, env.bfo_r, env.fth_r, env.fsh_r, env.ffo_r}')

time overall: 0:01:53.298370
height: 0.3583176344159821, width: 0.33114966511418087, spacing: 1.522659648453747
design: (0.5159883256649049, 0.7673906300598851, 0.5258731860322508, 0.5628558020478791, 0.8147063362295274, 0.7088109222827816)


In [7]:
import cv2


inference_fn = make_inference_fn(params)
jit_env_reset = jax.jit(env.reset)
jit_env_step = jax.jit(env.step)
jit_inference_fn = jax.jit(inference_fn)

rollout = []
rng = jax.random.PRNGKey(seed=1)
state = jit_env_reset(rng=rng)
for _ in range(episode_length):
  rollout.append(state.pipeline_state)
  act_rng, rng = jax.random.split(rng)
  act, _ = jit_inference_fn(state.obs, act_rng)
  state = jit_env_step(state, act)

frames = env.render(rollout, camera='track')
print(frames[0].shape)

# media.show_video(env.render(rollout, camera='side'), fps=1.0 / env.dt)
fps = 30  # Set the desired frames per second of the video
video_writer = cv2.VideoWriter(
    f'./logs/halfcheetah{strftime("%a, %d %b %Y %H:%M:%S +0000", gmtime())}.mp4', cv2.VideoWriter_fourcc(*"mp4v"), fps, (320, 240)  # remember to switch the x/y axis in cv2
)
for i in range(len(frames)):
    video_writer.write(cv2.cvtColor(frames[i], cv2.COLOR_RGB2BGR))
video_writer.release()

(240, 320, 3)


In [13]:
wandb.finish()

training/reward,▆█▁▃
training/reward,105.86212
